# Deep Learning Lab Practical 2
Handwritten Digit Recognition (MNIST)


In [2]:
pip install tensorflow

ERROR: Could not find a version that satisfies the requirement tensorflow (from versions: none)

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip3 install --upgrade pip
ERROR: No matching distribution found for tensorflow
Note: you may need to restart the kernel to use updated packages.


In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
import matplotlib.pyplot as plt
import pandas as pd


ModuleNotFoundError: No module named 'tensorflow'

In [ ]:

# Load MNIST dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train / 255.0
x_test = x_test / 255.0


In [ ]:

# Expand dimensions for CNN
x_train_cnn = x_train[..., None]
x_test_cnn = x_test[..., None]


In [ ]:

def build_cnn(activation='relu', optimizer='adam', dropout_rate=0.25, use_bn=False):
    model = models.Sequential()
    model.add(layers.Conv2D(32, (3,3), activation=activation, input_shape=(28,28,1)))
    model.add(layers.Conv2D(64, (3,3), activation=activation))
    model.add(layers.MaxPooling2D((2,2)))
    if use_bn:
        model.add(layers.BatchNormalization())
    model.add(layers.Dropout(dropout_rate))
    model.add(layers.Flatten())
    model.add(layers.Dense(128, activation=activation))
    model.add(layers.Dense(10, activation='softmax'))
    
    model.compile(optimizer=optimizer,
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model


In [ ]:

def build_mlp(units=[256,128], activation='relu', optimizer='adam', use_bn=True, dropout_rate=None):
    model = models.Sequential()
    model.add(layers.Flatten(input_shape=(28,28)))
    for u in units:
        model.add(layers.Dense(u))
        if use_bn:
            model.add(layers.BatchNormalization())
        model.add(layers.Activation(activation))
        if dropout_rate:
            model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(10, activation='softmax'))
    model.compile(optimizer=optimizer,
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model


In [ ]:

experiments = []

# CNN-1
cnn = build_cnn(activation='relu', optimizer='adam')
hist_cnn = cnn.fit(x_train_cnn, y_train, epochs=10, validation_split=0.1, verbose=0)
acc = cnn.evaluate(x_test_cnn, y_test, verbose=0)[1]
experiments.append(['CNN-1', 'ReLU', 'Adam', 10, acc])


In [ ]:

# MLP-1
mlp1 = build_mlp(units=[512,256,128], activation='relu', optimizer=optimizers.SGD())
hist_mlp1 = mlp1.fit(x_train, y_train, epochs=20, validation_split=0.1, verbose=0)
acc1 = mlp1.evaluate(x_test, y_test, verbose=0)[1]
experiments.append(['MLP-1', 'ReLU', 'SGD', 20, acc1])


In [ ]:

# MLP-2
mlp2 = build_mlp(units=[256], activation='relu', optimizer='adam')
hist_mlp2 = mlp2.fit(x_train, y_train, epochs=15, validation_split=0.1, verbose=0)
acc2 = mlp2.evaluate(x_test, y_test, verbose=0)[1]
experiments.append(['MLP-2', 'ReLU', 'Adam', 15, acc2])


In [ ]:

df = pd.DataFrame(experiments, columns=['Model','Activation','Optimizer','Epochs','Test Accuracy'])
df


In [ ]:

plt.plot(hist_cnn.history['accuracy'], label='CNN Train')
plt.plot(hist_cnn.history['val_accuracy'], label='CNN Val')
plt.legend()
plt.title("CNN Accuracy Curve")
plt.show()
